In [1]:
!pip install -q transformers datasets evaluate rouge-score sentencepiece

import os
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

print("✅ ደረጃ 1 ተጠናቋል: ሁሉም ቤተ-መጻሕፍት በተሳካ ሁኔታ ተጭነዋል!")

✅ ደረጃ 1 ተጠናቋል: ሁሉም ቤተ-መጻሕፍት በተሳካ ሁኔታ ተጭነዋል!


## Normalization

In [2]:
import re

def normalize_amharic(text):
    if not isinstance(text, str):
        return ""
    # ተመሳሳይ ድምፅ ያላቸውን የአማርኛ ፊደላት በአንድ መልክ ማቀናጀት
    text = re.sub(r'[ሐኃኅ]', 'ሀ', text)
    text = re.sub(r'[ሠ]', 'ሰ', text)
    text = re.sub(r'[ጸ]', 'ፀ', text)
    # ተጨማሪ ክፍተት (Spaces) ማስተካከል
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("✅ ደረጃ 2 ተጠናቋል: የአማርኛ ኖርማላይዜሽን (Normalization) ፈንክሽን ዝግጁ ነው!")

✅ ደረጃ 2 ተጠናቋል: የአማርኛ ኖርማላይዜሽን (Normalization) ፈንክሽን ዝግጁ ነው!


Load the Model and Tokenizer

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("🚀 Step 3: Downloading model and tokenizer in Google Colab... (Please wait a moment)")
MODEL_NAME = "google/mt5-small"

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

# Load the model (including the 1.2GB pytorch_model.bin file)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

print("✅ Model and tokenizer loaded successfully!")

🚀 Step 3: Downloading model and tokenizer in Google Colab... (Please wait a moment)


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


✅ Model and tokenizer loaded successfully!


Load the Amharic XL-Sum Dataset

In [4]:
!pip install -q datasets==2.19.0

In [5]:
from datasets import load_dataset

print("📊 ደረጃ 4: የአማርኛ XL-Sum ዳታሴት በመጫን ላይ...")

# trust_remote_code=True በመጠቀም ዳታሴቱን መጫን
dataset = load_dataset("csebuetnlp/xlsum", "amharic", trust_remote_code=True)

print("\n✅ ዳታሴቱ በተሳካ ሁኔታ ተጭኗል!")
print(dataset)

📊 ደረጃ 4: የአማርኛ XL-Sum ዳታሴት በመጫን ላይ...


Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]


✅ ዳታሴቱ በተሳካ ሁኔታ ተጭኗል!
DatasetDict({
    train: Dataset({
        features: ['id', 'url', 'title', 'summary', 'text'],
        num_rows: 5761
    })
    test: Dataset({
        features: ['id', 'url', 'title', 'summary', 'text'],
        num_rows: 719
    })
    validation: Dataset({
        features: ['id', 'url', 'title', 'summary', 'text'],
        num_rows: 719
    })
})


Preprocessing and Tokenizing the Dataset

In [6]:
# Function to preprocess and tokenize the examples
def preprocess_function(examples):
    # Input articles
    inputs = [str(ex) for ex in examples["text"]]
    # Target summaries
    targets = [str(ex) for ex in examples["summary"]]

    # Tokenize the inputs
    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding="max_length")

    # Tokenize the targets (summaries)
    labels = tokenizer(targets, max_length=128, truncation=True, padding="max_length")

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("🔄 Step 5: Processing the dataset in batches...")

# Map the preprocessing function across the entire dataset (Train and Test)
tokenized_datasets = dataset.map(preprocess_function, batched=True)

print("✅ Step 5 Complete: The data is now fully prepared for model training!")

🔄 Step 5: Processing the dataset in batches...


Map:   0%|          | 0/5761 [00:00<?, ? examples/s]

Map:   0%|          | 0/719 [00:00<?, ? examples/s]

Map:   0%|          | 0/719 [00:00<?, ? examples/s]

✅ Step 5 Complete: The data is now fully prepared for model training!


Model Training / Fine-Tuning

In [9]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

# 1. ዳታ ኮሌክተር ማዘጋጀት
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# 2. የስልጠና ፓራሜትሮችን ማስተካከል
training_args = Seq2SeqTrainingArguments(
    output_dir="./mt5_amharic_summarization",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=5e-4,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=False,
    logging_steps=100,
    save_steps=500
)

# 3. ትሬነሩን ማዘጋጀት (tokenizer=tokenizer ተወግዷል)
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
)

print("🚀 ደረጃ 6: የትሬኒንግ ሂደቱ በመጀመር ላይ ነው...")

# ስልጠናውን መጀመር
trainer.train()

print("🎉 ትሬኒንጉ በተሳካ ሁኔታ ተጠናቋል!")

🚀 ደረጃ 6: የትሬኒንግ ሂደቱ በመጀመር ላይ ነው...


Step,Training Loss
100,11.010651
200,3.529962
300,3.347610
400,2.724365
500,2.381449
600,2.206747
700,1.837016
800,1.666230
900,1.762248
1000,1.686270


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

🎉 ትሬኒንጉ በተሳካ ሁኔታ ተጠናቋል!


Testing / Inference

In [10]:
# ለመሞከር የሚሆን የአማርኛ ጽሑፍ
input_text = """
የኢትዮጵያ አየር መንገድ በአፍሪካ ግዙፍ ከሆኑ የአቪዬሽን ድርጅቶች አንዱ ሲሆን፡ በየጊዜው አዳዲስ አውሮፕላኖችን በማስገባት እና የበረራ መዳረሻዎቹን በማስፋፋት ይታወቃል።
አየር መንገዱ በቅርቡ ተጨማሪ ዘመናዊ አውሮፕላኖችን ለመግዛት ከቦይንግ ኩባንያ ጋር ስምምነት መፈረሙን አስታውቋል።
ይህ ስምምነት የአየር መንገዱን የወደፊት የዕድገት ጉዞ እና የአቪዬሽን መሪነቱን የበለጠ የሚያጠናክር ነው ተብሏል።
"""

# ጽሑፉን ወደ ቶከን መቀየር
inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True)

# ሞዴሉ ማጠቃለያ እንዲያመነጭ ማድረግ
summary_ids = model.generate(
    inputs["input_ids"].to(model.device),
    max_length=128,
    num_beams=4,
    early_stopping=True
)

# የነበሩትን ቶከኖች ወደ አማርኛ ጽሑፍ መመለስ
generated_summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("📝 **የመጀመሪያው ጽሑፍ:**\n", input_text)
print("\n✨ **በሞዴሉ የተዘጋጀው ማጠቃለያ:**\n", generated_summary)

📝 **የመጀመሪያው ጽሑፍ:**
 
የኢትዮጵያ አየር መንገድ በአፍሪካ ግዙፍ ከሆኑ የአቪዬሽን ድርጅቶች አንዱ ሲሆን፡ በየጊዜው አዳዲስ አውሮፕላኖችን በማስገባት እና የበረራ መዳረሻዎቹን በማስፋፋት ይታወቃል። 
አየር መንገዱ በቅርቡ ተጨማሪ ዘመናዊ አውሮፕላኖችን ለመግዛት ከቦይንግ ኩባንያ ጋር ስምምነት መፈረሙን አስታውቋል። 
ይህ ስምምነት የአየር መንገዱን የወደፊት የዕድገት ጉዞ እና የአቪዬሽን መሪነቱን የበለጠ የሚያጠናክር ነው ተብሏል።


✨ **በሞዴሉ የተዘጋጀው ማጠቃለያ:**
 የኢትዮጵያ አየር መንገድ በአፍሪካ ግዙፍ ስምምነት መፈረሙን አስታወቀች።
